# Notebook 4 — Inference Demo

**Sovereign Dialect-Bridge** — pipeline end-to-end:
teks dialek atau Bahasa Indonesia → *normalize* → *summarize* → ringkasan BI baku.

Evaluasi metrik (ROUGE/BERTScore pada 700 test samples) sudah dilakukan di .
Notebook ini hanya untuk demo interaktif dan verifikasi pipeline.


In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, re, time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Device: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
elif DEVICE == "cpu":
    print("[INFO] CPU — inference ~10-30 detik per teks.")


/opt/homebrew/Caskroom/miniforge/base/envs/ai_core/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : mps


In [2]:
import sys, importlib, subprocess

PACKAGES = {
    "transformers" : "transformers==4.40.0",
    "sentencepiece": "sentencepiece",
    "accelerate"   : "accelerate",
    "sklearn"      : "scikit-learn",
    "networkx"     : "networkx",
    "PySastrawi"   : "PySastrawi",
}

to_install = [pkg for mod, pkg in PACKAGES.items()
              if not importlib.util.find_spec(mod)]
if to_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + to_install, check=True)
    print(f"Installed: {to_install}")
else:
    print("Semua package sudah tersedia.")


Installed: ['PySastrawi']


In [3]:
CWD        = Path.cwd()
ROOT       = CWD if (CWD / "data").exists() else CWD.parent
MODELS_DIR = (ROOT / "model") if (ROOT / "model").exists() else (ROOT / "models")

NORM_DIR    = MODELS_DIR / "normalizer"
INDOT5_DIR  = MODELS_DIR / "indot5"
MT5BASE_DIR = MODELS_DIR / "mt5base"

GEN_KWARGS = dict(
    max_new_tokens       = 150,
    num_beams            = 2,    # 4→2: ~2x lebih cepat, drop ROUGE ~1-2 poin
    no_repeat_ngram_size = 3,
    early_stopping       = True,
    length_penalty       = 1.0,
)

print(f"ROOT       : {ROOT}")
print(f"MODELS_DIR : {MODELS_DIR}")
print()
for name, path in [("normalizer", NORM_DIR), ("indot5", INDOT5_DIR), ("mt5base", MT5BASE_DIR)]:
    print(f"  {name:12s} : {'Found' if path.exists() else 'Missing'}")


ROOT       : /Users/fadh/Documents/BINUS/Semester 4/Project_Summarize_NLP
MODELS_DIR : /Users/fadh/Documents/BINUS/Semester 4/Project_Summarize_NLP/model

  normalizer   : Found
  indot5       : Found
  mt5base      : Found


In [4]:
def clean_noise(text: str) -> str:
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_case_punct(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[!?]{2,}", "!", text)
    return re.sub(r"\s([.,!?;:])", r"\1", text).strip()


def preprocess_abstractive(text: str) -> str:
    return normalize_case_punct(clean_noise(text))


def split_sentences(text: str) -> list:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


print("Preprocessing helpers siap.")


Preprocessing helpers siap.


In [5]:
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from PySastrawi.Stemmer.StemmerFactory import StemmerFactory
except ImportError:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
_stemmer = StemmerFactory().create_stemmer()


def summarize_textrank(text: str, n_sentences: int = 3, max_words: int = 80) -> str:
    raw_sents  = split_sentences(text)
    stem_sents = split_sentences(_stemmer.stem(normalize_case_punct(clean_noise(text))))
    if len(raw_sents) <= n_sentences:
        # join dulu jadi string, baru potong per kata
        return " ".join(" ".join(raw_sents).split()[:max_words])
    try:
        vec    = TfidfVectorizer().fit_transform(stem_sents)
        sim    = cosine_similarity(vec, vec)
        np.fill_diagonal(sim, 0)
        scores = nx.pagerank(nx.from_numpy_array(sim))
        ranked = sorted(scores, key=scores.get, reverse=True)[:n_sentences]
        result = " ".join(raw_sents[i] for i in sorted(ranked))
    except Exception:
        result = " ".join(raw_sents[:n_sentences])
    return " ".join(result.split()[:max_words])


print("TextRank fallback siap.")


TextRank fallback siap.


In [6]:
# ── NER-based extractive summarizer ──────────────────────────────────────────
from transformers import pipeline as hf_pipeline

_ner_pipe = None

_ENTITY_LABELS = {
    "PER": "Orang",     "ORG": "Organisasi", "LOC": "Lokasi",
    "QTY": "Kuantitas", "TIM": "Waktu",       "EVT": "Kejadian",
    "LAW": "Regulasi",  "CRD": "Angka",       "NOR": "Lembaga",
}


def _get_ner_pipe():
    global _ner_pipe
    if _ner_pipe is None:
        _dev = 0 if DEVICE == "cuda" else ("mps" if DEVICE == "mps" else -1)
        try:
            _ner_pipe = hf_pipeline(
                "ner",
                model="cahya/bert-base-indonesian-NER",
                aggregation_strategy="simple",
                device=_dev,
            )
            print("  NER pipeline loaded.")
        except Exception as e:
            print(f"  [NER] load gagal: {e}")
    return _ner_pipe


def extract_entities(text: str) -> list:
    pipe = _get_ner_pipe()
    if pipe is None:
        return []
    try:
        raw = pipe(text[:512])
    except Exception:
        return []
    seen, result = set(), []
    for e in raw:
        word = e["word"].strip()
        key  = (word.lower(), e["entity_group"])
        if key not in seen and float(e["score"]) > 0.70:
            seen.add(key)
            result.append({
                "word" : word,
                "type" : e["entity_group"],
                "score": round(float(e["score"]), 3),
            })
    return sorted(result, key=lambda x: x["score"], reverse=True)


def summarize_ner(text: str, n_sentences: int = 3, max_words: int = 80) -> tuple:
    entities  = extract_entities(text)
    ent_words = {e["word"].lower() for e in entities}
    raw_sents = split_sentences(text)

    if len(raw_sents) <= n_sentences:
        joined = " ".join(raw_sents)
        return " ".join(joined.split()[:max_words]), entities

    scores  = [sum(1 for w in ent_words if w in s.lower()) for s in raw_sents]
    ranked  = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_sentences]
    summary = " ".join(raw_sents[i] for i in sorted(ranked))
    return " ".join(summary.split()[:max_words]), entities


def format_entities(entities: list) -> str:
    grouped = {}
    for e in entities:
        grouped.setdefault(e["type"], []).append(e["word"])
    if not grouped:
        return "(tidak ada)"
    return "  |  ".join(
        f"{_ENTITY_LABELS.get(t, t)}: {', '.join(w.title() for w in ws)}"
        for t, ws in grouped.items()
    )


print("NER summarizer siap.")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0602 17:33:29.524000 89097 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


NER summarizer siap.


In [7]:
def free_vram(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def _load_tokenizer(model_dir: Path, model_type: str = None):
    info_path = model_dir / "tokenizer_info.json"
    source    = str(model_dir)
    if info_path.exists():
        source = json.loads(info_path.read_text()).get("original_model", source)
        print(f"  tokenizer source: {source}")
    return AutoTokenizer.from_pretrained(source)


def _safe_decode(tok, ids) -> str:
    seq = ids.tolist() if hasattr(ids, "tolist") else list(ids)
    try:
        return tok.decode(seq, skip_special_tokens=True)
    except TypeError:
        return tok.decode(seq)


def _postprocess_output(text: str) -> str:
    """Bersihkan artifact mT5: merged CamelCase token dan kapitalisasi acak."""
    # "mencariJalan" → "mencari Jalan"
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r' +', ' ', text).strip()
    text = text.lower()
    # Kapitalisasi awal kalimat
    if text:
        text = text[0].upper() + text[1:]
    text = re.sub(r'([.!?] )([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    return text


print("Model helpers siap.")


Model helpers siap.


In [8]:
# Model cache: avoids reloading on repeated calls
_loaded_models: dict = {}

_SUMMARIZER_PRIORITY = [
    (MT5BASE_DIR, "mt5", "mT5-base"),
    (INDOT5_DIR,  "t5",  "IndoT5"),
]

# Minimum kata untuk menjalankan model neural; di bawah ini pakai TextRank
MIN_WORDS_NEURAL = 40


def _get_or_load_model(model_dir: Path, model_type: str) -> tuple:
    key = str(model_dir)
    if key not in _loaded_models:
        tok = _load_tokenizer(model_dir, model_type)
        mod = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir)).to(DEVICE).eval()
        _loaded_models[key] = (tok, mod)
        print(f"  Loaded: {model_dir.name}")
    return _loaded_models[key]


def unload_all():
    for k in list(_loaded_models.keys()):
        tok, mod = _loaded_models.pop(k)
        free_vram(tok, mod)
    print("Semua model di-unload.")


def _run_model(tok, model, text: str, model_type: str, max_input: int = 512) -> str:
    inp = preprocess_abstractive(text)
    if model_type == "mt5":
        inp = "summarize: " + inp
    enc = tok(inp, return_tensors="pt", truncation=True,
              max_length=max_input).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, **GEN_KWARGS)
    return _safe_decode(tok, out[0])


def run_pipeline(text: str, use_normalizer: bool = True) -> dict:
    """Pipeline satu model (prioritas mT5-base → IndoT5 → TextRank)."""
    t0   = time.time()
    n_in = len(text.split())

    normalized = text
    norm_used  = False
    qc_passed  = False
    if use_normalizer and NORM_DIR.exists():
        norm_tok, norm_mod = _get_or_load_model(NORM_DIR, "mt5")
        out_norm  = _run_model(norm_tok, norm_mod, text, "mt5", max_input=256)
        qc_passed = (len(out_norm.split()) / max(n_in, 1)) >= 0.30
        normalized = out_norm if qc_passed else text
        norm_used  = True

    summary    = None
    model_used = None
    if n_in >= MIN_WORDS_NEURAL:
        for model_dir, model_type, model_name in _SUMMARIZER_PRIORITY:
            if not model_dir.exists():
                continue
            sum_tok, sum_mod = _get_or_load_model(model_dir, model_type)
            summary    = _postprocess_output(_run_model(sum_tok, sum_mod, normalized, model_type))
            model_used = model_name
            break

    if summary is None:
        summary    = summarize_textrank(normalized)
        model_used = "TextRank" if n_in >= MIN_WORDS_NEURAL else f"TextRank (input < {MIN_WORDS_NEURAL} kata)"

    elapsed_ms    = round((time.time() - t0) * 1000)
    words_per_sec = round(n_in / max(elapsed_ms / 1000, 0.001))
    return {
        "input"            : text,
        "normalized"       : normalized if norm_used else None,
        "qc_passed"        : qc_passed  if norm_used else None,
        "summary"          : summary,
        "model_used"       : model_used,
        "compression_ratio": round(len(summary.split()) / max(n_in, 1), 3),
        "elapsed_ms"       : elapsed_ms,
        "words_per_sec"    : words_per_sec,
    }


def run_both_summarizers(text: str, use_normalizer: bool = False) -> dict:
    """Load mT5-base dan IndoT5, jalankan keduanya, return hasil masing-masing."""
    t0   = time.time()
    n_in = len(text.split())

    normalized = text
    norm_used  = False
    qc_passed  = False
    if use_normalizer and NORM_DIR.exists():
        norm_tok, norm_mod = _get_or_load_model(NORM_DIR, "mt5")
        out_norm  = _run_model(norm_tok, norm_mod, text, "mt5", max_input=256)
        qc_passed = (len(out_norm.split()) / max(n_in, 1)) >= 0.30
        normalized = out_norm if qc_passed else text
        norm_used  = True

    summaries = {}
    for model_dir, model_type, model_name, key in [
        (MT5BASE_DIR, "mt5", "mT5-base", "mt5base"),
        (INDOT5_DIR,  "t5",  "IndoT5",   "indot5"),
    ]:
        if not model_dir.exists():
            summaries[key] = None
            continue
        if n_in < MIN_WORDS_NEURAL:
            summaries[key] = {"model_name": model_name, "summary": None,
                               "note": f"input < {MIN_WORDS_NEURAL} kata, gunakan TextRank"}
            continue
        tok, mod  = _get_or_load_model(model_dir, model_type)
        clean     = _postprocess_output(_run_model(tok, mod, normalized, model_type))
        summaries[key] = {
            "model_name": model_name,
            "summary"   : clean,
            "n_words"   : len(clean.split()),
            "cr"        : round(len(clean.split()) / max(n_in, 1), 3),
        }

    elapsed_ms = round((time.time() - t0) * 1000)
    return {
        "input"     : text,
        "n_in"      : n_in,
        "normalized": normalized if norm_used else None,
        "qc_passed" : qc_passed  if norm_used else None,
        "elapsed_ms": elapsed_ms,
        **summaries,
    }


def _print_both(result: dict, label: str = ""):
    if label:
        print(f"INPUT {label} ({result['n_in']} kata):")
    print(f"  {result['input'][:200]}")
    if result["normalized"] is not None:
        qc = "lolos QC" if result["qc_passed"] else "QC gagal — pakai teks asli"
        print(f"NORMALIZED [{qc}]:")
        print(f"  {result['normalized'][:200]}")
    print()
    print("─── ABSTRACTIVE ───────────────────────────────────────────────────────────")
    for key in ("mt5base", "indot5"):
        r = result.get(key)
        if r is None:
            print(f"  [{key}] model tidak ditemukan")
        elif r.get("summary") is None:
            print(f"  [{r['model_name']}] {r.get('note', '-')}")
        else:
            print(f"  [{r['model_name']}] ({r['n_words']} kata, CR={r['cr']}):")
            print(f"    {r['summary']}")
        print()
    print(f"Total elapsed: {result['elapsed_ms']} ms")


print("Pipeline siap. Gunakan run_both_summarizers() untuk membandingkan mT5-base & IndoT5.")


Pipeline siap. Gunakan run_both_summarizers() untuk membandingkan mT5-base & IndoT5.


In [9]:
# ── Preload semua model + warmup (jalankan SEKALI sebelum demo) ───────────────
# Setelah cell ini selesai, semua demo cell berjalan tanpa delay loading.
import time as _t

_t0 = _t.time()
print("Preloading model ke cache...")

_models_to_load = [
    ("normalizer", NORM_DIR,    "mt5"),
    ("mT5-base",   MT5BASE_DIR, "mt5"),
    ("IndoT5",     INDOT5_DIR,  "t5"),
]
for _name, _dir, _type in _models_to_load:
    if _dir.exists():
        _get_or_load_model(_dir, _type)
        print(f"  [{round(_t.time()-_t0, 1)}s] {_name} loaded")
    else:
        print(f"  [skip] {_name} not found")

_get_ner_pipe()
print(f"  [{round(_t.time()-_t0, 1)}s] NER pipeline loaded")

# Warmup: satu inferensi dummy untuk pre-JIT MPS/CUDA kernel
_WARMUP = "Pemerintah akan meningkatkan anggaran pendidikan tahun ini."
if MT5BASE_DIR.exists():
    _tok_w, _mod_w = _loaded_models[str(MT5BASE_DIR)]
    _run_model(_tok_w, _mod_w, _WARMUP, "mt5")
    print(f"  [{round(_t.time()-_t0, 1)}s] Warmup selesai")

print(f"\nSiap! Total preload: {round(_t.time()-_t0, 1)}s — demo cell selanjutnya langsung cepat.")


Preloading model ke cache...


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 3460.28it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  Loaded: normalizer
  [11.0s] normalizer loaded


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1768.20it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  Loaded: mt5base
  [34.5s] mT5-base loaded


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 2101.99it/s]


  Loaded: indot5
  [40.0s] IndoT5 loaded


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36637.10it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  NER pipeline loaded.
  [63.5s] NER pipeline loaded
  [116.3s] Warmup selesai

Siap! Total preload: 116.3s — demo cell selanjutnya langsung cepat.


## Demo — Masukkan teks di cell berikut

Ganti isi  dengan teks yang ingin diringkas.
-  → aktifkan Stage 1 (cocok untuk teks dialek)
-  → langsung ke summarizer (cocok untuk teks BI baku)


In [10]:
# ── Contoh 1: Teks berita Bahasa Indonesia baku ──────────────────────────────
INPUT_TEXT = (
    "Badan Gizi Nasional (BGN) mengancam sanksi penghentian sementara operasional dapur Makan Bergizi Gratis (MBG) bagi Satuan Pelayanan Pemenuhan Gizi (SPPG) yang tidak memprioritaskan kelompok rentan, yakni ibu hamil, ibu menyusui, dan balita. "
    "Lewat Surat Edaran Nomor 5 Tahun 2026 yang dirilis Kedeputian Bidang Pemantauan dan Pengawasan (Tauwas) BGN, setiap dapur SPPG kini diwajibkan melayani minimal 300 penerima manfaat dari kelompok 3B, yakni ibu hamil, ibu menyusui, dan balita. "
    "Deputi Tauwas BGN Dadang Hendrayuda mengatakan aturan tersebut dibuat untuk memastikan kelompok paling rentan benar-benar mendapatkan akses program MBG. "
    "'Surat Edaran ini kami keluarkan untuk menjamin cakupan pelayanan gizi bagi kelompok 3B, dan meningkatkan konsistensi pelaksanaan SPPG di seluruh wilayah,' kata Dadang mengutip detikcom, Senin (25/5). "
    "Dadang mengungkapkan pihaknya masih menemukan banyak dapur MBG yang belum memenuhi target penerima manfaat dari kelompok 3B. "
    "Padahal sebelumnya BGN sudah menetapkan target pelayanan hingga 500 penerima manfaat untuk ibu hamil, ibu menyusui, dan balita. "
    "'Saat sidak di lapangan kami sering menemukan SPPG yang hanya melayani kurang dari 100 penerima manfaat 3B,' ujarnya. "
    "Lewat aturan terbaru ini, BGN menetapkan batas minimal baru yakni 300 penerima manfaat kelompok 3B di setiap SPPG. "
    "Selain itu, BGN memastikan akan menjatuhkan sanksi administratif bagi pihak yang tidak memenuhi ketentuan tersebut. "
    "Sementara bagi mitra maupun yayasan pengelola SPPG, sanksinya lebih berat. Dapur MBG yang tidak memenuhi kewajiban minimal pelayanan 3B akan dikenai suspend kategori major. "
    "'Karena sanksi yang dikenakan kepada mereka adalah suspend mayor, maka mereka tidak mendapatkan insentif Rp 6 juta per hari sampai pemenuhan ketentuan dapat dibuktikan,' kata Dadang. "
    "Berlaku 2 Juni 2026. Untuk pengawasan, Kepala SPPG diwajibkan menyampaikan laporan capaian pelayanan kelompok 3B secara berkala kepada Direktorat Wilayah Deputi Tauwas. "
    "Laporan tersebut nantinya akan diverifikasi dan menjadi dasar penilaian kepatuhan setiap dapur MBG. "
    "Meski begitu, Dadang menyebut pengelola tetap diberi kesempatan melakukan klarifikasi sesuai prosedur administratif BGN sebelum sanksi dijatuhkan. "
    "'Tapi yang jelas, aturan tentang pelayanan minimal 300 penerima manfaat dari kelompok 3B ini wajib dilaksanakan mulai tanggal 2 Juni 2026,' tegas purnawirawan jenderal Kopassus tersebut. "
    "BGN menilai kelompok ibu hamil, ibu menyusui, dan balita menjadi prioritas karena termasuk kelompok paling rentan mengalami masalah gizi dan stunting."
)

result            = run_both_summarizers(INPUT_TEXT, use_normalizer=False)
ner_sum, entities = summarize_ner(INPUT_TEXT)
tr_sum            = summarize_textrank(INPUT_TEXT)

print("─── EKSTRAKTIF ───────────────────────────────────────────────────────────")
print(f"Entitas  : {format_entities(entities)}")
print()
print(f"NER      ({len(ner_sum.split())} kata):")
print(f"  {ner_sum}")
print()
print(f"TextRank ({len(tr_sum.split())} kata):")
print(f"  {tr_sum}")
print()
_print_both(result, label="BERITA BI")


─── EKSTRAKTIF ───────────────────────────────────────────────────────────
Entitas  : Orang: Dadang Hend  |  Regulasi: Surat Edaran Nomor 5 Tahun 2026  |  Angka: 300  |  Lembaga: Deputi  |  Organisasi: Badan Gizi Nasional

NER      (80 kata):
  Badan Gizi Nasional (BGN) mengancam sanksi penghentian sementara operasional dapur Makan Bergizi Gratis (MBG) bagi Satuan Pelayanan Pemenuhan Gizi (SPPG) yang tidak memprioritaskan kelompok rentan, yakni ibu hamil, ibu menyusui, dan balita. Lewat Surat Edaran Nomor 5 Tahun 2026 yang dirilis Kedeputian Bidang Pemantauan dan Pengawasan (Tauwas) BGN, setiap dapur SPPG kini diwajibkan melayani minimal 300 penerima manfaat dari kelompok 3B, yakni ibu hamil, ibu menyusui, dan balita. Deputi Tauwas BGN Dadang Hendrayuda mengatakan aturan tersebut dibuat untuk memastikan kelompok

TextRank (32 kata):
  Badan Gizi Nasional (BGN) mengancam sanksi penghentian sementara operasional dapur Makan Bergizi Gratis (MBG) bagi Satuan Pelayanan Pemenuhan Gizi (SPPG)

In [11]:
# ── Contoh 2: Teks pengaduan dialek Sunda ─────────────────────────────────────
INPUT_DIALECT = (
    "Kunaon di Kopo teh sok macet wae jalanna. "
    "Pernah Urang ka kopo teh sok macet wae jalanna, utamana mun jam-jam rame. Urang mah bingung, naha teu bisa diatur ulang jalanna supaya teu macet wae? "
    "Mana jalanna sempit, terus di sisi jalanna aya warung-warung, jadi teu bisa dilewatan dua mobil pas papasan. Urang mah ngarepkeun bisa diatur ulang jalanna, misalna dijadikeun jalan searah atawa dilebarkeun jalanna. "
    "Terus mun keur hujan teh sok banjir dei, jadi mobil teu bisa lewat, terus macet deh jalanna. Urang mah ngarepkeun bisa diatur ulang jalanna, misalna dijadikeun jalan searah atawa dilebarkeun jalanna, terus ditambahin gorong-gorong biar mun hujan teu banjir dei. "
    "Teras di Kopo mah aneh da kereta whoosh ge kapergok macet, pesawat ge jeung nu laenna aduh bingung urang mah, kumaha mun jadi wargi kopo untung urang wargi cimahi. "
    "Pokona mah eta jalan kopo kudu dibenahkeun meh ngeunah lah tong sok macet wae jalanna."
)

result_dialect = run_both_summarizers(INPUT_DIALECT, use_normalizer=True)

text_for_ext = (
    result_dialect["normalized"]
    if result_dialect["normalized"] is not None and result_dialect["qc_passed"]
    else INPUT_DIALECT
)
ner_sum, entities = summarize_ner(text_for_ext)
tr_sum = summarize_textrank(text_for_ext)

print("─── EKSTRAKTIF ───────────────────────────────────────────────────────────")
print(f"Entitas  : {format_entities(entities)}")
print()
print(f"NER      ({len(ner_sum.split())} kata):")
print(f"  {ner_sum}")
print()
print(f"TextRank ({len(tr_sum.split())} kata):")
print(f"  {tr_sum}")
print()
_print_both(result_dialect, label="DIALEK SUNDA")


─── EKSTRAKTIF ───────────────────────────────────────────────────────────
Entitas  : Lokasi: Kopo Teh  |  Angka: Dua

NER      (38 kata):
  Kunaon di Kopo teh sok macet wae jalanna. Pernah Urang ka kopo teh sok macet wae jalanna, utamana mun jam-jam rame. Mana jalanna sempit, terus di sisi jalanna aya warung-warung, jadi teu bisa dilewatan dua mobil pas papasan.

TextRank (8 kata):
  Kunaon di Kopo teh sok macet wae jalanna.

INPUT DIALEK SUNDA (149 kata):
  Kunaon di Kopo teh sok macet wae jalanna. Pernah Urang ka kopo teh sok macet wae jalanna, utamana mun jam-jam rame. Urang mah bingung, naha teu bisa diatur ulang jalanna supaya teu macet wae? Mana jal
NORMALIZED [QC gagal — pakai teks asli]:
  Kunaon di Kopo teh sok macet wae jalanna. Pernah Urang ka kopo teh sok macet wae jalanna, utamana mun jam-jam rame. Urang mah bingung, naha teu bisa diatur ulang jalanna supaya teu macet wae? Mana jal

─── ABSTRACTIVE ───────────────────────────────────────────────────────────
  [mT5-base] (

In [12]:
# ── Contoh 3: Teks pengaduan dialek Minangkabau ──────────────────────────────
INPUT_MINANG = (
    "Ambo malaporan jalan di muko rumah ambo rusak bana, alah lamo indak dibeton. "
    "Urang nan lewat di jalan ko sering kacilakaan motonyo dek jalan nan balubuang gadang. "
    "Ambo harok supayo pemerintah daerah capek mambuek jalan tu elok baliak."
)

result_minang = run_both_summarizers(INPUT_MINANG, use_normalizer=True)

text_for_ext = (
    result_minang["normalized"]
    if result_minang["normalized"] is not None and result_minang["qc_passed"]
    else INPUT_MINANG
)
ner_sum, entities = summarize_ner(text_for_ext)
tr_sum = summarize_textrank(text_for_ext)

print("─── EKSTRAKTIF ───────────────────────────────────────────────────────────")
print(f"Entitas  : {format_entities(entities)}")
print()
print(f"NER      ({len(ner_sum.split())} kata):")
print(f"  {ner_sum}")
print()
print(f"TextRank ({len(tr_sum.split())} kata):")
print(f"  {tr_sum}")
print()
_print_both(result_minang, label="DIALEK MINANG")


─── EKSTRAKTIF ───────────────────────────────────────────────────────────
Entitas  : Lembaga: Pemerintah Daerah

NER      (24 kata):
  Saya melaporkan jalan di rumah saya rusak, saya lewat di jalan yang sering kecewa, tetapi harus pemerintah daerah harus mencari jalan yang tidak rusak.

TextRank (24 kata):
  Saya melaporkan jalan di rumah saya rusak, saya lewat di jalan yang sering kecewa, tetapi harus pemerintah daerah harus mencari jalan yang tidak rusak.

INPUT DIALEK MINANG (38 kata):
  Ambo malaporan jalan di muko rumah ambo rusak bana, alah lamo indak dibeton. Urang nan lewat di jalan ko sering kacilakaan motonyo dek jalan nan balubuang gadang. Ambo harok supayo pemerintah daerah c
NORMALIZED [lolos QC]:
  Saya melaporkan jalan di rumah saya rusak, saya lewat di jalan yang sering kecewa, tetapi harus pemerintah daerah harus mencari jalan yang tidak rusak.

─── ABSTRACTIVE ───────────────────────────────────────────────────────────
  [mT5-base] input < 40 kata, gunakan TextRank



In [13]:
# ── Input bebas — ganti teks di bawah ini ────────────────────────────────────
MY_TEXT = """
Tulis teks kamu di sini. Bisa berupa artikel berita, laporan pengaduan,
atau teks dalam dialek daerah. Pipeline akan otomatis memilih model terbaik.
"""

USE_NORMALIZER = True   # True jika teks dialek, False jika sudah BI baku

if MY_TEXT.strip() and "Tulis teks" not in MY_TEXT:
    out = run_pipeline(MY_TEXT.strip(), use_normalizer=USE_NORMALIZER)

    text_for_ext = (
        out["normalized"]
        if out["normalized"] is not None and out["qc_passed"]
        else MY_TEXT.strip()
    )
    ner_sum, entities = summarize_ner(text_for_ext)
    tr_sum = summarize_textrank(text_for_ext)

    print(f"INPUT ({len(MY_TEXT.split())} kata): {MY_TEXT[:150]}")
    if out["normalized"]:
        print(f"NORMALIZED : {out['normalized'][:150]}")
    print()
    print("─── EKSTRAKTIF ─────────────────────────────────────────────────────")
    print(f"Entitas  : {format_entities(entities)}")
    print(f"NER      : {ner_sum}")
    print(f"TextRank : {tr_sum}")
    print()
    print("─── NEURAL ──────────────────────────────────────────────────────────")
    print(f"[{out['model_used']}] CR={out['compression_ratio']}  |  {out['elapsed_ms']}ms ({out['words_per_sec']} kata/s)")
    print(f"  {out['summary']}")
else:
    print("Ganti MY_TEXT di atas dengan teks yang ingin diringkas.")


Ganti MY_TEXT di atas dengan teks yang ingin diringkas.
